# RNA3D thesis composite-TBM submission

Offline production inference using the same temporal-safe/composite-search code as the local experiments.

In [ ]:
from pathlib import Path
import os
import shutil
import sys
import tarfile
import time

import pandas as pd

ARTIFACT_DATASET = Path('/kaggle/input/rna3d-thesis-inference-artifacts')
ARCHIVE = ARTIFACT_DATASET / 'rna3d_bundle.tar.gz'
BUNDLE = ARTIFACT_DATASET
WORK = Path('/kaggle/working/rna3d_work')
COMPETITION = Path('/kaggle/input/stanford-rna-3d-folding')

if not (BUNDLE / 'bundle_manifest.json').is_file():
    assert ARCHIVE.is_file(), sorted(str(p) for p in Path('/kaggle/input').glob('*'))
    BUNDLE = Path('/kaggle/working/rna3d_bundle')
    if BUNDLE.exists():
        shutil.rmtree(BUNDLE)
    BUNDLE.mkdir(parents=True)
    with tarfile.open(ARCHIVE, 'r:gz') as archive:
        archive.extractall(BUNDLE)
sys.path.insert(0, str(BUNDLE))
sys.path.insert(0, str(BUNDLE / 'src'))
RUNTIME_MMSEQS = Path('/kaggle/working/mmseqs')
shutil.copy2(BUNDLE / 'bin/mmseqs', RUNTIME_MMSEQS)
RUNTIME_MMSEQS.chmod(0o755)
os.environ['LD_LIBRARY_PATH'] = str(BUNDLE / 'lib')
print('bundle ready:', BUNDLE, 'files:', len(list(BUNDLE.rglob('*'))))

In [ ]:
from kaggle.inference_pipeline import run_inference
from rna3d.template import mmseqs_search
mmseqs_search.mmseqs_bin = lambda: str(RUNTIME_MMSEQS)

test = pd.read_csv(COMPETITION / 'test_sequences.csv')
sample = pd.read_csv(COMPETITION / 'sample_submission.csv')
started = time.time()
submission = run_inference(
    test,
    BUNDLE,
    work_dir=WORK,
    sample_submission=sample,
    steps=300,
    max_len=1000,
)
output = Path('/kaggle/working/submission.csv')
submission.to_csv(output, index=False)
assert submission.shape == sample.shape, (submission.shape, sample.shape)
assert submission['ID'].tolist() == sample['ID'].tolist()
assert not submission.isna().any().any()
print(f'wrote {output}: rows={len(submission)}, targets={len(test)}, seconds={time.time()-started:.1f}')

In [ ]:
submission.head(3)